# RAG and Agentic Memory on Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/rag-agent-memory/rag-agent-memory-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

A vector store does exactly one thing: it holds a pile of vectors and finds the ones nearest a query vector. No reasoning, nothing that understands your question, nothing that can decline to answer.

Almost everything called "RAG" or "agent memory" is that one operation with different plumbing around it. The difference between them is not the lookup — it is **who holds the pen**:

|  | RAG | Agentic memory |
|---|---|---|
| Who writes | a pipeline, ahead of time | the agent, as it goes |
| What's in it | documents you chose | facts the agent noticed |
| How it's read | embed query → k nearest | embed query → k nearest |

That last row is identical, which is why both live in this one notebook. Steps 0–3 build a read-only index. Step 4 changes who writes to it, and nothing else changes.

**Steps 0, 1 and 3 never call a model**, so you can get to a working retrieval demo — with a picture — before you find your API key.

---

## What is RAG, concretely?

**The problem.** A language model only knows its training data. It has never seen your
docs, your notes, or anything written after its cutoff. Fine-tuning on your documents is
expensive and goes stale; pasting *all* of them into the prompt doesn't fit.

**The trick.** Before the model answers, go find the three or four most relevant
paragraphs and paste *those* in. **Retrieval** finds the paragraphs; **generation** is the
model answering with them in front of it. That's the whole idea.

It happens at two different times, and mixing them up is the usual source of confusion:

**Ahead of time — build the index** (step 0, once):

```
documents  →  chunks  →  embeddings  →  vector store
  ~50 docs     ~550 pieces   ~550 vectors    Chroma
  READMEs     ~1200 chars   384 numbers     on disk
```

**At question time — retrieve, then answer** (steps 1 and 2, every question):

```
"How do I use GRPO?"
        ↓ embed the question the same way
   384 numbers
        ↓ find the nearest vectors
   top 4 chunks
        ↓ paste into the prompt      ← the "augmented" part
   model answers, citing them
```

### Three words

- **Embedding** — a list of numbers standing for a piece of text, arranged so similar text lands near similar text. Here: `bge-small`, 384 numbers, runs on CPU, no API key. Closeness is **cosine similarity** — in this corpus a good match scores ~0.75+, an unrelated one ~0.45.
- **Chunk** — documents are split into ~1200-character pieces, because one vector has to stand for one chunk. A chunk spanning four topics averages them into mush. This is the most consequential knob in the pipeline.
- **Vector store** — the thing holding the vectors that answers "which are nearest?". Here it's Chroma: a sqlite file on disk.

### What actually reaches the model

No framework magic. Step 2 builds this string and sends it:

```
CONTEXT:
[#1] (source: tutorials/code-mode-analysis/README.md)
A code-mode agent does something different. Given the same tools, it writes...

[#2] (source: tutorials/code-mode-analysis/README.md)
Flyte runs the generated program in Monty, a Rust-based Python interpreter...

QUESTION: What does the code-mode tutorial teach?
```

...plus a system prompt saying *answer only from the context, cite chunks as [#N], say so if it isn't there.* That is RAG in full. Everything else here is about making the **retrieval** half good, because the generation half is just this.

### And agentic memory?

Same lookup, pointed the other way. After each exchange the agent asks a model *"did the user reveal anything durable?"*, gets back something like `["The user's name is Sage."]`, embeds those sentences and stores them. Next turn it embeds your new message and retrieves the nearest stored facts.

That's why step 4's agent answers "what do you know about me?" with no special handling for that question — it isn't replaying the conversation, it's retrieving sentences that earlier turns wrote, because your question landed near them.

---

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/rag-agent-memory
    !uv pip install --system -r requirements.txt
    !uv pip install --system keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

from utils.file_viewer import view_file
from utils.report_viewer import show_latest

### Set your Anthropic key

Steps 2, 4 and 5 call a model. **Steps 0, 1 and 3 do not** — run those with no key at all.

Locally you can put `ANTHROPIC_API_KEY=sk-ant-...` in a `.env` file instead; `config.py` loads it for you.

In [ ]:
# Skip this if the key is already in a .env file or your environment.
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

# Want a cheaper model for a workshop? Or a local one?
# os.environ["LLM_MODEL"] = "claude-haiku-4-5"
# os.environ["LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"   # Ollama

In [ ]:
# Optional: connect to a Flyte or Union cluster.
# Skip this to run everything locally — every step below works with no cluster.
#
# Don't have one? Request demo access at https://union.ai/
#
# !flyte create config \
#     --endpoint <your-endpoint> \
#     --project flytesnacks \
#     --domain development \
#     --builder remote
#
# Then create the secret the tasks read, in the SAME project and domain:
# !flyte create secret ANTHROPIC_API_KEY -p flytesnacks -d development

### Running the steps

Every step is a plain `flyte run`, so the usual flags work:

| | |
|---|---|
| `flyte run --local ...` | Runs on this machine. No cluster, no image build. Still gets caching and reports. |
| `flyte run --local --tui ...` | Same, with the live task UI. |
| `flyte run ...` | Runs on the cluster: containers, retries, and the run graph in the UI. |

The cells below use `--local` so the notebook works with no cluster at all.

Each step calls the ones before it as subtasks. Those are cached, so **only the first run pays for building the index** — everything after starts instantly.

### 0. Build the index

This is the **ahead-of-time** phase. Three tasks: fetch documents, split them into chunks, embed the chunks into a Chroma collection.

Why split at all? Because one embedding has to represent one chunk. Embed a whole 24KB README as a single vector and it means "something about Flyte" — too vague to retrieve usefully. Embed ~1200-character pieces and each vector means one specific thing.

Why embed? Because that's what makes "find me relevant text" a *math* problem: similar text becomes nearby numbers, and "nearby" is something a database can look up in milliseconds.

The collection comes out as a `flyte.io.Dir` — one artifact that every later step takes as an input.

The default corpus is **this repository's own tutorial write-ups**: around 50 documents and 550 chunks, a few seconds to embed on CPU. Good for a workshop because you can check every answer by opening the file it cites.

No API key needed for this one.

In [ ]:
!flyte run --local step0_index.py index --store_backend qdrant

#### Which vector store is this using?

**Qdrant** — every cell in this notebook passes `--store_backend qdrant`, so you are running the swappable path rather than reading about it.

`store.py` supports two backends behind one interface, and both are file-backed: no server, no API key, nothing to sign up for. Drop the flag anywhere to get **Chroma**, which is the default:

```
!flyte run --local step0_index.py index                          # chroma
!flyte run --local step0_index.py index --store_backend qdrant   # qdrant
```

They produce **identical rankings and identical scores** — same encoder, same cosine similarity, so they should. That equality is the useful part: it means the abstraction isn't lying.

| | #1 | #2 | #3 | #4 |
|---|---|---|---|---|
| chroma | 0.802 | 0.795 | 0.771 | 0.748 |
| qdrant | 0.802 | 0.795 | 0.771 | 0.748 |

Keep the flag consistent across cells. `store_backend` is part of the Flyte cache key, so mixing them builds two separate indexes — and each store directory records which engine wrote it, so pointing the wrong backend at one gives you a clear error instead of silently zero results.

The interface is five operations — `count`, `add`, `nearest`, `all_records`, and opening one. That's genuinely everything a RAG pipeline asks of a vector database. Adding pgvector or LanceDB means writing one class and changing no step files.

Two gotchas, both living in `nearest()`: Chroma returns cosine *distance* (0 = identical) while Qdrant returns a *score* (1.0 = identical), so each backend normalizes to "similarity, higher is better" — get it backwards and you silently retrieve the **worst** matches with no error anywhere. And Qdrant needs int/UUID ids, so `QdrantStore` hashes `tutorials/x.md::3` into a stable UUID and keeps the original in the payload.

*(Qdrant Cloud is deliberately not wired up: a task writing to a remote database has no `flyte.io.Dir` to hand the next step, so artifact chaining and caching both break. Embedded mode keeps all of it — and keeps this notebook credential-free.)*

In [ ]:
# Other corpora, same command:
#   --source flyte-docs                       the Flyte OSS docs (~8MB)
#   --source hf --dataset_repo <hf-dataset>   any HuggingFace text dataset
#   --source local --local_path ~/notes       your own markdown

view_file("step0_index.py", title="fetch, chunk, embed", show_path=True)

### 1. Search it, with no model anywhere

This is the **retrieval** half of retrieval-augmented generation, on its own.

Embed the question with the same encoder that embedded the documents, ask the store for the nearest vectors. That's the entire operation — no model, no reasoning, no API key. Just: which of these ~550 vectors is closest to this one?

Everything people find disappointing about RAG in production is a failure *here*, not in the model. Which is why it's worth looking at alone before anything can hide it.

In [ ]:
!flyte run --local step1_retrieve.py search --store_backend qdrant \
    --question "How do I fine-tune a model with GRPO?"

In [ ]:
show_latest()

Now ask it something the corpus has never heard of.

Watch the similarity scores, not the text.

In [ ]:
!flyte run --local step1_retrieve.py search --store_backend qdrant \
    --question "Who won the 2022 FIFA World Cup?"

In [ ]:
show_latest()

**You still got four chunks.**

That is the most important thing to understand about retrieval, and it's why this step comes before the model. Retrieval has no concept of "I don't know" — it returns the nearest neighbours whether or not they're any good. The similarity scores are the only signal you get. In-corpus questions land around 0.78; that last one was 0.47.

If nothing downstream thresholds on that number, nothing downstream can tell the difference.

In [ ]:
# `retrieve()` is 15 lines and there is no magic in them.
view_file("store.py", title="chunking, embedding, and the lookup", show_path=True)

### 2. Answer from the chunks

Now the **generation** half. Take the chunks step 1 found, paste them into the prompt, ask the model to cite them as `[#N]` and to refuse when they don't cover the question.

The literal string sent to the model:

```
CONTEXT:
[#1] (source: tutorials/code-mode-analysis/README.md)
<the text of chunk 1>

[#2] (source: ...)
<the text of chunk 2>

QUESTION: What does the code-mode tutorial teach?
```

That's it. RAG is not an architecture — it's a prompt with freshly-retrieved text in it. Everything hard about it happened upstream, in what you chunked and whether the neighbours were any good.

In [ ]:
!flyte run --local step2_rag_answer.py answer --store_backend qdrant \
    --question "What does the code-mode tutorial teach?"

In [ ]:
# Read the citations here, not in the terminal above — Flyte's console renders
# with Rich, which treats [#1] as markup and swallows it.
show_latest()

Now the same question with retrieval switched off.

Flyte turns a `bool` task parameter into a click flag pair, so it's `--no-use_retrieval`. (`--use_retrieval false` fails with `Got unexpected extra argument (false)`.)

In [ ]:
!flyte run --local step2_rag_answer.py answer --store_backend qdrant \
    --question "What does the code-mode tutorial teach?" --no-use_retrieval

In [ ]:
show_latest()

You'll get one of two things, and which one is luck. Sometimes a confident answer about a completely different "code mode" — the agentic-coding-tool persona, or the Cloudflare MCP pattern. Sometimes a hedge that lists a few things it might be and asks you to narrow it down.

The hedge looks like the model behaving well, and in a sense it is. But notice what you *still* can't do with either answer: check it. There's nothing to open, no claim tied to anything.

The grounded run's `[#3]` points at a file on disk. That's what retrieval buys — not confidence, **verifiability**.

In [ ]:
view_file("step2_rag_answer.py", title="retrieve, stuff the prompt, answer", show_path=True)

# One file holds every model call, so switching to OpenAI or a local Ollama
# server is an environment variable rather than an edit.
view_file("llm.py", title="the only place a model is called", show_path=True)

### 3. Look at the space

Every chunk is a 384-dimensional vector. UMAP squashes that down to two so it fits on a screen, keeping neighbours near neighbours. The corpus becomes a map with clusters nobody labelled.

Your question goes through the *same fitted* projection and lands as an orange star, labelled with the question itself.

**Reading the chart:** small gray dots are every chunk in the index. **Numbered blue dots** are the retrieved chunks — *darker means a better match*, and the number is the rank. The orange star is your question.

Rank is an ordered quantity, so it gets a single-hue ramp rather than a rainbow — a red→green scale would imply red is bad, when red would be the *best* match. Adjacent ramp steps are necessarily similar, so the rank number is drawn inside each dot and color never carries rank alone.

Two dots landing on top of each other isn't a glitch: consecutive chunks from one document have nearly identical embeddings, so you'll often see three dots when four were retrieved. That's chunk size made visible — the legend always lists all of them.

No API key needed here either. The first run takes ~20 seconds while numba compiles; after that the fit is cached and questions are instant.

In [ ]:
!flyte run --local step3_visualize.py visualize --store_backend qdrant \
    --question "How do I fine-tune a model with GRPO?"

In [ ]:
show_latest()

A completely different corner of the corpus:

In [ ]:
!flyte run --local step3_visualize.py visualize --store_backend qdrant \
    --question "brain tumor segmentation"

In [ ]:
show_latest()

And the question the corpus can't answer.

**Read the neighbourhood, not the distance.** UMAP has to put an out-of-corpus question *somewhere*, and it will drop it next to whatever is least unlike it — so a lonely-looking star is not the tell. The tell is that the highlighted chunks have nothing to do with each other or with what you asked, and the scores are low. The numbers are ground truth; the map shows which neighbourhood they came from.

In [ ]:
!flyte run --local step3_visualize.py visualize --store_backend qdrant \
    --question "Who won the 2022 FIFA World Cup?"

In [ ]:
show_latest()

In [ ]:
# The projection is fitted once and cached — deliberately. Refit per question and
# the whole cloud reshuffles between runs, which makes it unreadable. You want the
# map to hold still while the star moves.
view_file("step3_visualize.py", title="fit once, project the query, draw", show_path=True)

### 4. Turn the store around

Same Chroma, same encoder, same nearest-neighbour lookup. The only difference: **the agent writes to it.**

Each turn does four things:

1. embed the message, retrieve the most relevant memories
2. answer, with those memories in the system prompt
3. a second, cheap model call extracts durable facts from the exchange as JSON
4. embed those facts and write them back

The default script is three messages: introduce yourself, mention a constraint, then ask what it knows.

In [ ]:
!flyte run --local step4_memory.py converse --store_backend qdrant

In [ ]:
show_latest()

Turn 3 has **no special handling**. It recalls turn 1 because turn 1 put something in the store that turn 3's question is near. That is the entire mechanism behind "the agent remembers me."

Memory comes back as a `flyte.io.Dir`, so it outlives the run. Feed it to another `converse` and the agent picks up where it left off — copy the path printed above into the cell below.

In [ ]:
# Paste the memory directory from the run above:
# !flyte run --local step4_memory.py converse --store_backend qdrant \
#     --memory_dir <path-from-above> \
#     --messages '["Remind me what my time limit is and who I am."]'

#### How does it decide what to store?

**No entity recognition, no NER, no knowledge graph.** One extra model call per turn, with a prompt and a schema — that's the entire mechanism. The prompt (`EXTRACTION_SYSTEM`) says:

> You extract durable facts about the user from one exchange. Return each fact as a short, self-contained sentence that will still make sense months from now, read on its own with no surrounding conversation.
>
> **Include:** stable preferences, constraints, decisions, roles, projects, identity. **Exclude:** questions, small talk, anything about the assistant, anything true only right now.

Three things make it work: the output is **schema-constrained** to `{"facts": [...]}` so there's no regex fishing JSON out of prose; "**self-contained sentence**" matters because a memory is retrieved later with none of its conversation around it; and **near-duplicates are dropped** at 0.95 cosine so five phrasings of one fact don't crowd out the top-k.

#### Where it breaks

Run this and watch it fail — you contradict yourself, and *both* facts get stored:

In [ ]:
!flyte run --local step4_memory.py converse --store_backend qdrant --messages '[ \
  "I am Sage and I always run my demos in Python.", \
  "Actually, I switched everything over to Rust last month.", \
  "What language do I use?"]'

In [ ]:
# Look at the store directly — nothing was updated or deleted.
import glob, sys
sys.path.insert(0, ".")
from store import open_collection

latest = max(glob.glob(".rag_work/memory_*"), key=lambda p: __import__("os").path.getmtime(p))
c = open_collection(latest, "agent_memory")
d = c.get(include=["documents", "metadatas"])
for doc, m in sorted(zip(d["documents"], d["metadatas"]), key=lambda x: x[1]["source"]):
    print(f"[{m['source']}] {doc}")

The agent still *answers* correctly — but notice why: retrieval handed the model both the old and the new fact, and the model reasoned its way to the right answer at read time. The memory itself is contradictory. Let the store grow until the stale fact outranks the fresh one and that stops working.

Three gaps, all fixable, none fixed here:

- **No entity resolution.** "Sage", "the user" and "their" are unrelated sentences. Nothing links them — which is also why this is single-user.
- **No conflict resolution.** Dedupe catches near-*duplicates*. Two facts that *contradict* aren't similar in vector space — same topic, opposite content — so a 0.95 threshold sails straight past them.
- **No usable timestamps.** Metadata says `turn 1`, `turn 2`, and the counter restarts every run, so you can't tell today's fact from last month's.

Real systems add an update/delete path (retrieve related memories first, ask the model whether the new fact supersedes one), entity IDs, and wall-clock timestamps. That's the natural next thing to build.

In [ ]:
# Two load-bearing details: extraction is schema-constrained (no regex fishing a
# {...} block out of prose), and near-duplicate memories are dropped so five
# phrasings of the same fact don't crowd out the top-k.
view_file("step4_memory.py", title="retrieve, answer, extract, write back", show_path=True)

### 5. Put it together

Chat on the left, live projection on the right, tabs for retrieved chunks and current memories. Every message retrieves, answers, moves the star, and writes what it learned about you.

**It does not rebuild the index.** Step 0's tasks are cached, so the app resolves the same Chroma directory in about a second and embeds nothing — it only builds one if you skipped step 0 entirely. (If you ran step 0 with a different `--source`, pass the same one here, or you'll build a second index from the default corpus.)

In Colab you need `--share` to get a reachable URL. Locally, drop it and open `http://localhost:7860`. The cell blocks while the app runs — stop it to carry on.

In [ ]:
# Uncomment to launch. Startup fits UMAP once, so give it ~30 seconds.
# !python step5_chat_app.py --local --share --store qdrant

On a cluster, this deploys as a real app instead:

```bash
python step5_chat_app.py
```

It mounts the index through `flyte.app.RunOutput`, so the app pod downloads the artifact step 0 already produced rather than rebuilding it.

---

## What you built

- A document index as a cached, versioned Flyte artifact
- Retrieval you can inspect, with the scores that tell you when to distrust it
- Grounded answers whose citations point at files you can open
- A picture of the embedding space, and an honest read of what it does and doesn't show
- An agent that writes to the same store it reads from

## What this deliberately doesn't do

So nobody mistakes this for production RAG:

- **No evaluation.** Nothing here tells you whether retrieval is *good*, only what it returned. Biggest gap by far.
- **A single dense lookup.** No re-ranking, no keyword search, no query rewriting.
- **Naive chunking.** Fixed character counts, ignoring headings, code blocks and tables.
- **Memory is single-user and never forgets.**

You can see it in the output above: ask *"How do I use GRPO?"* and `detr-object-detection` shows up at #2. Small corpus plus a bare dense lookup will do that.

## The rest of the RAG landscape

Roughly in order of value-for-effort. Nearly all of it changes only `store.retrieve()` or `step0_index.py`.

**Make retrieval better**
- **Re-ranking** — retrieve 20 with the fast bi-encoder, re-score with a *cross-encoder* that reads query and chunk together, keep 4. Usually the single biggest win.
- **Hybrid search** — dense embeddings miss exact tokens (error codes, flag names); BM25 keyword search nails those and misses paraphrase. Run both, merge with Reciprocal Rank Fusion. Their failure modes barely overlap.
- **Contextual retrieval** — prepend an LLM-written sentence situating each chunk in its document *before* embedding it. Fixes chunk-lost-its-context at the root.
- **Query rewriting / HyDE** — search with a rewritten query, or with a *hypothetical answer* the model writes, on the theory that a fake answer resembles a real one more than the question does.
- **Metadata filtering** — filter by source/date/type before the vector search. This tutorial already stores `source`.
- **Smarter chunking** — sentence-window or parent-document: embed something small for precision, return something big for context.

**Different shapes**
- **Hierarchical / tree (RAPTOR)** — recursively cluster and summarize into a tree, retrieve at any level. Flat top-k structurally cannot answer "what are the main themes across all ~50 documents?" — no single chunk holds that. A tree can.
- **Graph RAG** — entities and relations in a knowledge graph, traversed for multi-hop questions ("which tutorials use the same base model as the GRPO one?") where the answer spans chunks that never co-occur.
- **Multi-vector / ColBERT** — a vector per token instead of per chunk. More precise, much more storage.

**Different control flow**
- **Agentic RAG** — hand the model `retrieve` as a *tool* and let it decide whether to search, what to search for, and whether to search again. Multi-hop and "I don't know yet" fall out for free.
- **Corrective RAG** — grade the chunks before answering; re-query or refuse if they're weak. Cheap version using a number you already have: if top similarity < 0.5, don't answer.

**Measure it** — a golden set of ~50 question → correct-chunk pairs, then track **recall@k** and **MRR**. That's the number that moves when you add re-ranking. Plus faithfulness/relevance on the answers (RAGAS, or an LLM judge). The unglamorous version — 20 questions, run after every change, read them yourself — catches most regressions.

**When not to use RAG** — if the whole corpus fits in context, just paste it in. If the question is "how many X", that's SQL, not similarity. If you want a different writing style, that's fine-tuning.

## Concrete next steps for this code

`embed_and_index` is the only task that knows what Chroma is — point it at pgvector or LanceDB and nothing else changes. Beyond that: threshold on similarity so the World Cup case fails honestly; add a cross-encoder re-ranker; give memories an `entity_id` for multi-user; add a scheduled task that prunes memories nothing has retrieved in a month.